# Reproduce the ARMD susceptibility mart with Python

Transform the original cultures CSV into the same cleaned data source used
by Tableau, following the independently validated R pipeline. This notebook
uses **pandas method chains**, named intermediate data frames, and eight
sequential sections. The shared dbt seed CSVs supply reporting policy;
no database connection or execution of the R code is required.

Save this file as `notebooks/build_mart_susceptibility.ipynb` in your repository.
Set the paths in the first code cell, then **Restart Kernel and Run All**.
Use a Jupyter Python kernel with pandas 2.2 or later and NumPy installed.
If needed, run `%pip install pandas numpy` in a separate cell, then restart
the kernel. Each section depends on the preceding sections.

**Execution status:** this copy contains unexecuted code cells. The original
2,241,050-row cultures CSV was not included among the supplied files, so
a full Python run on that source has not yet been verified. The expected
checkpoints below are from the successful R run and supplied Tableau CSV;
they are not presented as Python execution results. The Python logic was
tested separately using small regression fixtures.

| Checkpoint | Expected for the supplied release and 2023 window |
|---|---:|
| Source rows | 2,241,050 |
| Consolidated cultures | 751,075 |
| Cohort culture–original-organism observations | 133,246 |
| Reviewed observations eligible for reporting | 126,405 |
| Original organism observations in the reporting window | 10,862 |
| Consolidated culture–organism–phenotype groups | 10,835 |
| First stratified observations | 9,203 |
| First overall *S. aureus* observations | 500 |
| Final mart rows / columns / reporting populations | 396 / 26 / 73 |
| Nonkey field values compared / expected differences | 8,316 / 0 |

These are source organism observations, not verified biological isolate
identifiers. Dates are shifted research timestamps. This notebook reproduces
the project's supplied reporting definitions and does not reinterpret AST
using new clinical breakpoints.


## 1. Import and stage the original data

Keep identifiers as text and blank CSV fields as missing values. Preserve
raw susceptibility text before trimming. In particular, a literal `Null`
becomes missing **before** trimming susceptibility, matching the R code.
Other cleaned text fields are trimmed before handling `Null`.


In [ ]:
from pathlib import Path
import hashlib
import re
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# Detect the repository from either its root or a notebooks/ subfolder.
project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "dbt" / "seeds").is_dir()),
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
)

# Edit these paths to point to your existing files; do not duplicate raw data.
raw_data_file = project_root / "data/raw/microbiology_cultures_cohort.csv"
seed_dir = project_root / "dbt/seeds"
tableau_reference_file = project_root / "data/exports/mart_susceptibility.csv"
output_file = project_root / "data/processed/mart_susceptibility_python.csv"

source_release = "2025-04-11"
analysis_window_start = pd.Timestamp("2023-01-01", tz="UTC")
analysis_window_end_exclusive = analysis_window_start + pd.DateOffset(months=12)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70)
print(f"pandas {pd.__version__}; NumPy {np.__version__}")


### Shared helpers

These small helpers keep the transformations readable. `sql_merge()` prevents
missing identifiers from matching one another. This is needed because the
default [pandas merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)
can match null keys. `anti_join()` returns left-side rows with no matching
key. `make_dbt_key()` uses the same ordered, hyphen-separated text and MD5
hash as the project's dbt keys.


In [ ]:
def read_text_csv(path):
    return pd.read_csv(
        path, dtype="string", keep_default_na=False, na_values=[""],
        skipinitialspace=False,
    )


def clean_text(series):
    return series.str.strip().replace("Null", pd.NA)


def sql_merge(left, right, on, how="inner", validate=None):
    keys = [on] if isinstance(on, str) else list(on)
    if how not in {"inner", "left"}:
        raise ValueError("sql_merge supports inner and left joins.")
    return left.merge(
        right.dropna(subset=keys), on=keys, how=how,
        validate=validate, sort=False,
    )


def anti_join(left, right, on):
    keys = [on] if isinstance(on, str) else list(on)
    marker = "_matched_right_key"
    matches = right[keys].dropna().drop_duplicates().assign(**{marker: True})
    return (
        sql_merge(left, matches, on=keys, how="left", validate="many_to_one")
        .loc[lambda x: x[marker].isna()]
        .drop(columns=marker)
        .reset_index(drop=True)
    )


def make_dbt_key(data, key_columns):
    if data.empty:
        return pd.Series(index=data.index, dtype="string")
    key_text = (
        data[key_columns].astype("string")
        .fillna("_dbt_utils_surrogate_key_null_")
        .agg("-".join, axis=1)
    )
    return key_text.map(lambda value: hashlib.md5(value.encode("utf-8")).hexdigest())


def row_counts(data, columns, name="row_count"):
    return (
        data.groupby(columns, dropna=False, observed=True, sort=False)
        .size().rename(name).reset_index()
    )


In [ ]:
raw_armd = (
    read_text_csv(raw_data_file)
    .rename(columns=lambda name: re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_"))
)

required_columns = [
    "anon_id", "pat_enc_csn_id_coded", "order_proc_id_coded",
    "order_time_jittered_utc", "ordering_mode", "culture_description",
    "was_positive", "organism", "antibiotic", "susceptibility",
]
missing_columns = sorted(set(required_columns) - set(raw_armd.columns))
if missing_columns:
    raise ValueError(f"Expected source columns not found: {missing_columns}")

armd_staged = pd.DataFrame({
    "source_version": source_release,
    "patient_key": raw_armd["anon_id"],
    "encounter_key": raw_armd["pat_enc_csn_id_coded"],
    "culture_order_key": raw_armd["order_proc_id_coded"],
    "order_time_jittered_utc_raw": raw_armd["order_time_jittered_utc"],
    "culture_time_shifted": pd.to_datetime(
        raw_armd["order_time_jittered_utc"], format="mixed", utc=True, errors="coerce"
    ),
    "ordering_mode": clean_text(raw_armd["ordering_mode"]),
    "culture_description": clean_text(raw_armd["culture_description"]),
    "was_positive": pd.to_numeric(
        clean_text(raw_armd["was_positive"]), errors="coerce"
    ).astype("Int64"),
    "organism": clean_text(raw_armd["organism"]),
    "antibiotic": clean_text(raw_armd["antibiotic"]),
    "susceptibility_raw": raw_armd["susceptibility"],
    "susceptibility": raw_armd["susceptibility"].replace("Null", pd.NA).str.strip(),
})

import_check = pd.DataFrame([{
    "source_rows": len(armd_staged),
    "timestamp_parse_failures": int((
        armd_staged["order_time_jittered_utc_raw"].notna()
        & ~armd_staged["order_time_jittered_utc_raw"].isin(["", "Null"])
        & armd_staged["culture_time_shifted"].isna()
    ).sum()),
    "missing_patient_keys": int(armd_staged["patient_key"].isna().sum()),
    "missing_order_keys": int(armd_staged["culture_order_key"].isna().sum()),
}])

display(import_check, armd_staged.head())
display(row_counts(armd_staged, ["culture_description"]).sort_values("row_count", ascending=False))
if import_check["timestamp_parse_failures"].iloc[0] != 0:
    raise ValueError("Review unparsed nonmissing timestamps before continuing.")


## 2. Consolidate cultures and reconcile source AST

A culture is identified by release, patient, encounter, and order. Preserve
an order-level field only when its source values agree, counting missing
values as a distinct state. One conflicting field flags the culture but
does not automatically exclude it.

AST is reconciled within the original organism label. Repeated identical
results resolve; different results, including a result mixed with missing
data, conflict. Built-in group operations keep this step practical for the
full source file. `first()` is only retained when the distinct-state check
permits it, so its missing-value skipping cannot hide a conflict.


In [ ]:
culture_id = ["source_version", "patient_key", "encounter_key", "culture_order_key"]
organism_id = culture_id + ["organism"]
culture_fields = [
    "culture_time_shifted", "ordering_mode", "culture_description", "was_positive"
]

culture_groups = armd_staged.groupby(
    culture_id, dropna=False, observed=True, sort=False
)
culture_field_states = culture_groups[culture_fields].nunique(dropna=False)

cultures = (
    culture_groups[culture_fields].first()
    .where(culture_field_states.eq(1))
    .join(culture_groups.size().rename("source_row_count"))
    .assign(has_order_level_conflict=culture_field_states.gt(1).any(axis=1))
    .reset_index()
    .assign(cohort_id=lambda x: x["culture_description"].map({
        "BLOOD": "blood", "URINE": "urine", "RESPIRATORY": "respiratory"
    }).astype("string"))
)

culture_organisms = (
    armd_staged.loc[lambda x: x["organism"].notna(), organism_id]
    .drop_duplicates()
    .pipe(sql_merge, cultures, on=culture_id, how="inner", validate="many_to_one")
)

ast_source = (
    armd_staged.loc[lambda x: x["organism"].notna() & x["antibiotic"].notna()]
    .assign(result_is_missing=lambda x: x["susceptibility"].isna().astype("int64"))
)
ast_groups = ast_source.groupby(
    organism_id + ["antibiotic"], dropna=False, observed=True, sort=False
)

ast_endpoints = (
    ast_groups.agg(
        source_result_rows=("susceptibility", "size"),
        missing_result_rows=("result_is_missing", "sum"),
        candidate_result=("susceptibility", "first"),
    )
    .join(ast_groups["susceptibility"].nunique(dropna=False).rename("result_state_count"))
    .reset_index()
    .assign(resolution_status=lambda x: np.select(
        [x["result_state_count"].gt(1), x["candidate_result"].isna()],
        ["conflict", "missing"], default="resolved"
    ))
    .assign(
        susceptibility=lambda x: x["candidate_result"].where(x["resolution_status"].eq("resolved")),
        repeated_result_rows=lambda x: x["source_result_rows"] - x["result_state_count"],
    )
    .drop(columns="candidate_result")
    .pipe(sql_merge, culture_organisms[organism_id], on=organism_id,
          how="inner", validate="many_to_one")
)

culture_check = pd.DataFrame([{
    "culture_count": len(cultures),
    "source_rows_accounted_for": int(cultures["source_row_count"].sum()),
    "cultures_with_conflicting_fields": int(cultures["has_order_level_conflict"].sum()),
    "missing_encounter_keys": int(cultures["encounter_key"].isna().sum()),
    "missing_culture_times": int(cultures["culture_time_shifted"].isna().sum()),
}])
ast_check = row_counts(ast_endpoints, ["resolution_status"], "endpoint_count")
display(culture_check, ast_check)
display(row_counts(culture_organisms, ["cohort_id"], "culture_organism_count"))


## 3. Apply reviewed organism mappings

Join the same organism dictionary and Gram-group reference tables used by
R and dbt. Preserve the original organism label so phenotype evidence is
still evaluated separately before labels are combined. Keep unmatched
observations available for review.


In [ ]:
organism_mapping = (
    read_text_csv(seed_dir / "organism_mapping.csv")
    .assign(
        organism_reporting_group=lambda x: x["organism_reporting_group"].str.strip().replace("", pd.NA),
        source_qualifier=lambda x: x["source_qualifier"].str.strip().replace("", pd.NA),
        include_primary_summary=lambda x: pd.to_numeric(x["include_primary_summary"], errors="raise").astype("Int64"),
        has_reviewed_mapping=True,
    )
    .rename(columns={"organism_source": "organism"})
)
organism_reporting_groups = read_text_csv(seed_dir / "organism_reporting_groups.csv")

mapped_organisms = (
    culture_organisms.loc[lambda x: x["cohort_id"].notna()]
    .pipe(sql_merge, organism_mapping, on="organism", how="left", validate="many_to_one")
    .assign(
        has_reviewed_mapping=lambda x: x["has_reviewed_mapping"].eq(True),
        mapping_status=lambda x: x["mapping_status"].where(x["has_reviewed_mapping"], "unmapped"),
        include_primary_summary=lambda x: x["include_primary_summary"].fillna(0).astype("Int64"),
    )
    .pipe(sql_merge, organism_reporting_groups, on="organism_reporting_group",
          how="left", validate="many_to_one")
)

unmapped_labels = (
    mapped_organisms.loc[lambda x: ~x["has_reviewed_mapping"]]
    .pipe(row_counts, ["organism"], "culture_organism_count")
    .sort_values("culture_organism_count", ascending=False)
)
mapping_check = pd.DataFrame([{
    "culture_organism_records": len(mapped_organisms),
    "distinct_source_labels": mapped_organisms["organism"].nunique(dropna=False),
    "matched_records": int(mapped_organisms["has_reviewed_mapping"].sum()),
    "unmapped_records": int((~mapped_organisms["has_reviewed_mapping"]).sum()),
    "primary_reporting_records": int(mapped_organisms["include_primary_summary"].eq(1).sum()),
    "primary_records_missing_gram": int((
        mapped_organisms["include_primary_summary"].eq(1)
        & mapped_organisms["gram_stain_group"].isna()
    ).sum()),
}])
display(mapping_check, unmapped_labels.head(20))


## 4. Classify resistance phenotypes from AST evidence

Evaluate CRE, extended-spectrum cephalosporin resistance (ESCE-R), oxacillin resistance in *S. aureus*, vancomycin resistance in *E. faecalis*/*E. faecium*, and multidrug resistance in *P. aeruginosa*. These are the existing project's definitions, reproduced from the validated R pipeline.

Keep the culture and **original organism label** as the classification grain. Source-name qualifiers remain available for tracing observations; the classifications themselves use AST evidence.


### 4.1 Select the reviewed scope and applicable rules

Apply the reviewed reporting-scope filter before classification. Each original organism observation is evaluated independently, so excluded observations cannot change retained observations' evidence. Unmatched labels remain in the earlier mapping audit.

For the supplied dictionary's 41 mapped source labels, standardized-name eligibility agrees with the SQL's standardized-name and source-label fallback checks. Recheck that equivalence if the dictionary is extended.


In [ ]:
rule_columns = [
    "cre_standard", "cre_proteus_morganella", "esce", "oxacillin", "vre", "mdr"
]

organism_scope = (
    mapped_organisms
    .loc[lambda x: (
        x["include_primary_summary"].eq(1)
        & x["mapping_status"].eq("reviewed")
        & x["organism_reporting_group"].notna()
    ).fillna(False)]
    .assign(standardized_name=lambda x: x["organism_standardized"].fillna(""))
    .assign(
        cre_standard=lambda x: (
            x["standardized_name"].str.match(
                r"^(Citrobacter |Enterobacter |Klebsiella )", na=False
            )
            | x["standardized_name"].isin(["Escherichia coli", "Serratia marcescens"])
        ),
        cre_proteus_morganella=lambda x: (
            x["standardized_name"].str.startswith("Proteus ", na=False)
            | x["standardized_name"].eq("Morganella morganii")
        ),
        esce=lambda x: (
            x["standardized_name"].isin([
                "Escherichia coli", "Klebsiella aerogenes", "Klebsiella oxytoca",
                "Klebsiella pneumoniae"
            ])
            | x["standardized_name"].str.startswith("Enterobacter ", na=False)
        ),
        oxacillin=lambda x: x["standardized_name"].eq("Staphylococcus aureus"),
        vre=lambda x: x["standardized_name"].isin([
            "Enterococcus faecalis", "Enterococcus faecium"
        ]),
        mdr=lambda x: x["standardized_name"].eq("Pseudomonas aeruginosa"),
    )
    .drop(columns="standardized_name")
)

rule_eligibility = (
    organism_scope[organism_id + rule_columns]
    .melt(id_vars=organism_id, value_vars=rule_columns,
          var_name="rule", value_name="eligible")
    .loc[lambda x: x["eligible"].fillna(False)]
    .drop(columns="eligible")
    .reset_index(drop=True)
)


### 4.2 Define the antibiotics used by each rule

Split the compact drug lists into one row per antibiotic. Proteus/Morganella CRE classification excludes imipenem-based evidence. The six-category MDR definition uses tobramycin alone for its aminoglycoside category.


In [ ]:
phenotype_panels = (
    pd.DataFrame([
        ("cre_standard", "cre",
         "Ertapenem|Imipenem|Meropenem|Meropenem/Vaborbactam|Imipenem/Relebactam"),
        ("cre_proteus_morganella", "cre",
         "Ertapenem|Meropenem|Meropenem/Vaborbactam"),
        ("esce", "esce",
         "Cefepime|Ceftriaxone|Cefotaxime|Ceftazidime|Ceftazidime/Avibactam|Ceftolozane/Tazobactam"),
        ("oxacillin", "oxacillin", "Oxacillin|Cefoxitin"),
        ("vre", "vre", "Vancomycin"),
        ("mdr", "es_ceph",
         "Cefepime|Ceftazidime|Ceftazidime/Avibactam|Ceftolozane/Tazobactam"),
        ("mdr", "fluoroquinolone", "Ciprofloxacin|Levofloxacin"),
        ("mdr", "aminoglycoside", "Tobramycin"),
        ("mdr", "carbapenem", "Imipenem|Meropenem|Imipenem/Relebactam"),
        ("mdr", "piperacillin_tazobactam", "Piperacillin/Tazobactam"),
        ("mdr", "cefiderocol", "Cefiderocol"),
    ], columns=["rule", "category", "antibiotic"])
    .assign(antibiotic=lambda x: x["antibiotic"].str.split("|", regex=False))
    .explode("antibiotic", ignore_index=True)
)


### 4.3 Summarize observed AST evidence

A categorized endpoint is resolved and contains Susceptible, Intermediate, or Resistant. Resistant is positive evidence for the four ordinary rules; Intermediate or Resistant is positive evidence for MDR. Drugs can contribute to more than one rule, so the first join intentionally permits multiple matches.

Only observed endpoints contribute here. An untested drug is not an invalid result; drugs outside these panels remain available for later susceptibility reporting.


In [ ]:
rule_evidence = (
    ast_endpoints
    .pipe(sql_merge, phenotype_panels, on="antibiotic", how="inner",
          validate="many_to_many")
    .pipe(sql_merge, rule_eligibility, on=organism_id + ["rule"], how="inner",
          validate="many_to_one")
    .assign(
        is_categorized=lambda x: (
            x["resolution_status"].eq("resolved")
            & x["susceptibility"].isin(["Susceptible", "Intermediate", "Resistant"])
        ).fillna(False),
        is_positive=lambda x: (
            x["susceptibility"].isin(["Resistant"])
            | (x["rule"].eq("mdr") & x["susceptibility"].isin(["Intermediate"]))
        ).fillna(False),
    )
    .assign(
        positive=lambda x: x["is_categorized"] & x["is_positive"],
        nonpositive=lambda x: x["is_categorized"] & ~x["is_positive"],
        invalid=lambda x: ~x["is_categorized"],
    )
    .groupby(organism_id + ["rule", "category"],
             as_index=False, observed=True, dropna=False, sort=False)
    .agg(
        observed_endpoint_count=("antibiotic", "size"),
        positive_count=("positive", "sum"),
        nonpositive_count=("nonpositive", "sum"),
        invalid_count=("invalid", "sum"),
    )
)


### 4.4 Assign an evidence status to each applicable category

Create the required categories, including all six MDR categories. With no observed endpoint, a category is `not_tested`. Positive evidence takes precedence over invalid results; invalid results otherwise take precedence over observed nonpositive evidence.


In [ ]:
required_categories = phenotype_panels[["rule", "category"]].drop_duplicates()
evidence_count_columns = [
    "observed_endpoint_count", "positive_count", "nonpositive_count", "invalid_count"
]

phenotype_category_status = (
    rule_eligibility
    .pipe(sql_merge, required_categories, on="rule", how="inner",
          validate="many_to_many")
    .pipe(sql_merge, rule_evidence, on=organism_id + ["rule", "category"],
          how="left", validate="one_to_one")
    .assign(**{
        column: (lambda x, column=column: x[column].fillna(0).astype("int64"))
        for column in evidence_count_columns
    })
    .assign(category_status=lambda x: np.select(
        [x["positive_count"].gt(0), x["invalid_count"].gt(0),
         x["nonpositive_count"].gt(0)],
        ["positive", "not_evaluable", "observed_non_positive"],
        default="not_tested",
    ))
)


### 4.5 Derive the five phenotype classifications

For ordinary phenotypes, positive evidence produces `positive`; observed nonpositive evidence without invalid endpoints produces `not_detected`. Otherwise, retain `not_evaluable`.

For MDR *P. aeruginosa*, three positive categories establish `positive`. If the positive and unknown categories together cannot reach three, classify the observation as `not_detected`; otherwise use `not_evaluable`. Both invalid and untested categories are unknown. Organisms outside a rule's scope receive `not_applicable`.


In [ ]:
ordinary_phenotype_status = (
    phenotype_category_status
    .loc[lambda x: x["rule"].ne("mdr")]
    .assign(
        phenotype=lambda x: x["category"].map({
            "cre": "ast_cre_status",
            "esce": "ast_esce_enterobacterales_status",
            "oxacillin": "ast_oxacillin_resistance_status",
            "vre": "ast_vancomycin_resistance_status",
        }),
        status=lambda x: (
            x["category_status"]
            .map({"positive": "positive", "observed_non_positive": "not_detected"})
            .fillna("not_evaluable")
        ),
    )
    [organism_id + ["phenotype", "status"]]
)

mdr_summary = (
    phenotype_category_status
    .loc[lambda x: x["rule"].eq("mdr")]
    .assign(
        positive_category=lambda x: x["category_status"].eq("positive"),
        unknown_category=lambda x: x["category_status"].isin([
            "not_evaluable", "not_tested"
        ]),
    )
    .groupby(organism_id, as_index=False, observed=True, dropna=False, sort=False)
    .agg(
        mdr_positive_category_count=("positive_category", "sum"),
        mdr_unknown_category_count=("unknown_category", "sum"),
    )
    .assign(ast_mdr_p_aeruginosa_status=lambda x: np.select(
        [x["mdr_positive_category_count"].ge(3),
         (x["mdr_positive_category_count"] + x["mdr_unknown_category_count"]).lt(3)],
        ["positive", "not_detected"],
        default="not_evaluable",
    ))
)

phenotype_columns = [
    "ast_cre_status",
    "ast_esce_enterobacterales_status",
    "ast_oxacillin_resistance_status",
    "ast_vancomycin_resistance_status",
    "ast_mdr_p_aeruginosa_status",
]

phenotype_status_long = pd.concat([
    ordinary_phenotype_status,
    mdr_summary.assign(
        phenotype="ast_mdr_p_aeruginosa_status",
        status=lambda x: x["ast_mdr_p_aeruginosa_status"],
    )[organism_id + ["phenotype", "status"]],
], ignore_index=True)

phenotype_status = (
    phenotype_status_long
    .pivot(index=organism_id, columns="phenotype", values="status")
    .reindex(columns=phenotype_columns)
    .fillna("not_applicable")
    .rename_axis(columns=None)
    .reset_index()
)

phenotype_organisms = (
    organism_scope
    .pipe(sql_merge, phenotype_status, on=organism_id,
          how="left", validate="one_to_one")
    .assign(**{
        column: (lambda x, column=column: x[column].fillna("not_applicable"))
        for column in phenotype_columns
    })
)


### 4.6 Review the phenotype counts

**Expected checkpoint from the validated R run:** 126,405 culture–organism observations, 41 distinct source labels, and zero records missing a phenotype status. The following cells calculate the Python results from the actual input; no checkpoint is prefilled as an execution result.

These are source observations before the reporting window, label consolidation, or first-culture selection.


In [ ]:
phenotype_population_check = pd.DataFrame([{
    "culture_organism_records": len(phenotype_organisms),
    "distinct_source_labels": phenotype_organisms["organism"].nunique(dropna=False),
    "missing_status_records": int(phenotype_organisms[phenotype_columns].isna().any(axis=1).sum()),
}])

phenotype_check = (
    phenotype_organisms[phenotype_columns]
    .melt(var_name="phenotype", value_name="status")
    .groupby(["phenotype", "status"], as_index=False,
             observed=True, dropna=False, sort=False)
    .size()
    .rename(columns={"size": "culture_organism_count"})
    .pivot(index="phenotype", columns="status", values="culture_organism_count")
    .fillna(0)
    .astype("int64")
    .rename_axis(columns=None)
    .reset_index()
)

display(phenotype_population_check)
display(phenotype_check)


The resulting `phenotype_organisms` frame is ready for assigning reporting categories and consolidating labels. CRE and MDR classifications are retained because they determine which observations may enter the main susceptibility-reporting populations.


## 5. Build reporting cultures and reconcile their AST

Assign each original organism observation to a reporting phenotype, apply the reporting period, and combine source labels within the same culture, organism group, and phenotype. The `base` category means that no positive phenotype took precedence; it does not establish susceptibility.

Overall *S. aureus* has a separate culture-level population that combines labels across phenotypes. Its results must be reconciled separately.


### 5.1 Assign reporting categories and apply the reporting period

The category priority matches the validated R and SQL: CRE, MDR *P. aeruginosa*, oxacillin resistance, vancomycin resistance, ESCE-R, then `base`. The start is inclusive and the end is exclusive, using UTC timestamps.


In [ ]:
window_organisms = (
    phenotype_organisms
    .assign(
        reporting_phenotype=lambda d: np.select(
            [
                d["ast_cre_status"].eq("positive").fillna(False).to_numpy(dtype=bool),
                d["ast_mdr_p_aeruginosa_status"].eq("positive").fillna(False).to_numpy(dtype=bool),
                d["ast_oxacillin_resistance_status"].eq("positive").fillna(False).to_numpy(dtype=bool),
                d["ast_vancomycin_resistance_status"].eq("positive").fillna(False).to_numpy(dtype=bool),
                d["ast_esce_enterobacterales_status"].eq("positive").fillna(False).to_numpy(dtype=bool),
            ],
            ["cre", "mdr_p_aeruginosa", "oxacillin_resistant", "vancomycin_resistant", "esce_r"],
            default="base",
        )
    )
    .loc[lambda d: (
        d["culture_time_shifted"].notna()
        & d["culture_time_shifted"].ge(analysis_window_start)
        & d["culture_time_shifted"].lt(analysis_window_end_exclusive)
    )]
    .copy()
)

reporting_culture_id = culture_id + [
    "cohort_id", "organism_reporting_group", "reporting_phenotype"
]
overall_culture_id = culture_id + ["cohort_id", "organism_reporting_group"]


### 5.2 Combine source labels within each reporting scope

Retain the original source names as lists for traceability. A numeric rank implements the evidence precedence `positive` > `not_evaluable` > `not_detected` > `not_applicable`; taking the maximum rank avoids a Python function call for every status group.


In [ ]:
phenotype_status_rank = {
    "not_applicable": 0,
    "not_detected": 1,
    "not_evaluable": 2,
    "positive": 3,
}
phenotype_status_label = {rank: label for label, rank in phenotype_status_rank.items()}


def consolidate_reporting_cultures(data, group_columns, include_source_phenotypes=False):
    ranked = data.assign(**{
        column: data[column].map(phenotype_status_rank)
        for column in phenotype_columns
    })
    aggregation = {
        "culture_time_shifted": ("culture_time_shifted", "first"),
        "gram_stain_group": ("gram_stain_group", "first"),
        "source_label_count": ("organism", "nunique"),
        "source_labels": ("organism", lambda x: sorted(x.dropna().unique().tolist())),
        **{column: (column, "max") for column in phenotype_columns},
    }
    if include_source_phenotypes:
        aggregation["source_reporting_phenotypes"] = (
            "reporting_phenotype", lambda x: sorted(x.dropna().unique().tolist())
        )
    combined = (
        ranked
        .groupby(group_columns, dropna=False, sort=False, observed=True, as_index=False)
        .agg(**aggregation)
    )
    return combined.assign(**{
        column: combined[column].map(phenotype_status_label)
        for column in phenotype_columns
    })


reporting_cultures = consolidate_reporting_cultures(
    window_organisms, reporting_culture_id
)
overall_cultures = consolidate_reporting_cultures(
    window_organisms.loc[
        lambda d: d["organism_reporting_group"].eq("Staphylococcus aureus")
    ],
    overall_culture_id,
    include_source_phenotypes=True,
)


### 5.3 Reconcile AST across original labels

Stage 2 resolved repeated results within one original organism label. This function combines those endpoints across labels in the reporting group. Any source conflict or disagreement between resolved results produces conflict. One consistent resolved result is retained even if another source label has a missing endpoint. An absent endpoint contributes no row.

| Source endpoints | Reporting result |
|---|---|
| Susceptible + Susceptible | Resolved Susceptible |
| Susceptible + missing | Resolved Susceptible |
| Susceptible + Resistant | Conflict |
| Any existing conflict | Conflict |
| Missing only | Missing |


In [ ]:
def reconcile_reporting_ast(data, group_columns):
    keys = group_columns + ["antibiotic"]
    combined = (
        data
        .assign(
            source_conflict=lambda d: d["resolution_status"].eq("conflict").astype("int64"),
            source_missing=lambda d: d["resolution_status"].eq("missing").astype("int64"),
            resolved_result=lambda d: d["susceptibility"].where(
                d["resolution_status"].eq("resolved")
            ),
        )
        .groupby(keys, dropna=False, sort=False, observed=True, as_index=False)
        .agg(
            source_endpoint_count=("resolution_status", "size"),
            source_conflict_count=("source_conflict", "sum"),
            source_missing_count=("source_missing", "sum"),
            distinct_resolved_result_count=("resolved_result", "nunique"),
            candidate_result=("resolved_result", "first"),
        )
        .assign(
            resolution_status=lambda d: np.select(
                [
                    d["source_conflict_count"].gt(0)
                    | d["distinct_resolved_result_count"].gt(1),
                    d["distinct_resolved_result_count"].eq(1),
                ],
                ["conflict", "resolved"],
                default="missing",
            ),
            susceptibility=lambda d: d["candidate_result"].where(
                d["resolution_status"].eq("resolved")
            ),
            reconciliation_reason=lambda d: np.select(
                [
                    d["source_conflict_count"].gt(0),
                    d["distinct_resolved_result_count"].gt(1),
                    d["distinct_resolved_result_count"].eq(1)
                    & d["source_missing_count"].gt(0),
                    d["distinct_resolved_result_count"].eq(1),
                ],
                [
                    "source_endpoint_conflict",
                    "discordant_source_results",
                    "resolved_with_missing_source_endpoint",
                    "resolved_consistent_result",
                ],
                default="missing_only",
            ),
        )
        .drop(columns="candidate_result")
    )
    return combined


### 5.4 Keep stratified and overall reconciliation separate

Attach the reporting identities using all original culture and organism keys. Stratified AST stays within the source phenotype. Overall *S. aureus* AST combines all of its labels on that culture. MSSA will later be derived from qualifying `base` *S. aureus* cultures.


In [ ]:
window_ast = sql_merge(
    ast_endpoints,
    window_organisms.loc[:, organism_id + [
        "cohort_id", "organism_reporting_group", "reporting_phenotype"
    ]],
    on=organism_id,
    how="inner",
    validate="many_to_one",
)

stratified_ast = (
    window_ast
    .loc[lambda d: d["reporting_phenotype"].isin([
        "base", "esce_r", "oxacillin_resistant", "vancomycin_resistant"
    ])]
    .pipe(reconcile_reporting_ast, reporting_culture_id)
)
overall_ast = (
    window_ast
    .loc[lambda d: d["organism_reporting_group"].eq("Staphylococcus aureus")]
    .pipe(reconcile_reporting_ast, overall_culture_id)
)


### 5.5 Inspect the reporting populations

These views overlap, so their counts are not additive totals of distinct patients or cultures. All tables below are generated by running the notebook.


In [ ]:
reporting_scope_check = pd.DataFrame({
    "population": [
        "Original organism observations in the reporting window",
        "Consolidated culture-organism-phenotype groups",
        "Overall Staphylococcus aureus cultures",
    ],
    "record_count": [len(window_organisms), len(reporting_cultures), len(overall_cultures)],
})
reporting_ast_check = (
    pd.concat([
        stratified_ast.groupby("resolution_status", dropna=False).size()
        .rename("endpoint_count").reset_index().assign(scope="stratified"),
        overall_ast.groupby("resolution_status", dropna=False).size()
        .rename("endpoint_count").reset_index().assign(scope="overall_s_aureus"),
    ], ignore_index=True)
    .loc[:, ["scope", "resolution_status", "endpoint_count"]]
    .sort_values(["scope", "resolution_status"])
    .reset_index(drop=True)
)
reporting_culture_counts = (
    reporting_cultures
    .groupby(["cohort_id", "reporting_phenotype"], dropna=False)
    .size().rename("culture_group_count").reset_index()
)
display(reporting_scope_check, reporting_culture_counts, reporting_ast_check)


## 6. Select the first qualifying culture within each category

Each patient contributes at most one culture per source release, cohort, organism reporting group, and published category. Categories are selected independently: a patient may contribute to both MRSA and MSSA, and to the separate overall *S. aureus* population.

### 6.1 Create deterministic culture ordering keys

Sort by shifted culture time and then the same MD5 culture key used by dbt and R. The helper defined earlier joins the four culture identifiers in their original order before hashing.


In [ ]:
culture_keys = (
    window_organisms.loc[:, culture_id]
    .drop_duplicates()
    .assign(culture_key=lambda d: make_dbt_key(d, culture_id))
)
stratified_sources = (
    sql_merge(
        reporting_cultures,
        culture_keys,
        on=culture_id,
        how="left",
        validate="many_to_one",
    )
    .assign(source_reporting_phenotype=lambda d: d["reporting_phenotype"])
)


### 6.2 Identify MSSA candidates

MSSA requires a `base` *S. aureus* culture, oxacillin-resistance status `not_detected`, and at least one observed oxacillin or cefoxitin endpoint. Every observed marker endpoint must be resolved Susceptible. Missing, conflicting, Intermediate, or Resistant markers prevent qualification.

Preserve `source_reporting_phenotype = "base"` for joining AST, while publishing the category as `oxacillin_susceptible`.


In [ ]:
mssa_marker_cultures = (
    stratified_ast
    .loc[lambda d: (
        d["organism_reporting_group"].eq("Staphylococcus aureus")
        & d["reporting_phenotype"].eq("base")
        & d["antibiotic"].isin(["Oxacillin", "Cefoxitin"])
    )]
    .assign(marker_is_susceptible=lambda d: (
        d["resolution_status"].eq("resolved")
        & d["susceptibility"].isin(["Susceptible"])
    ).fillna(False))
    .groupby(reporting_culture_id, dropna=False, sort=False, observed=True, as_index=False)
    .agg(all_observed_markers_susceptible=("marker_is_susceptible", "all"))
    .loc[lambda d: d["all_observed_markers_susceptible"], reporting_culture_id]
)

mssa_candidates = (
    stratified_sources
    .loc[lambda d: (
        d["organism_reporting_group"].eq("Staphylococcus aureus")
        & d["reporting_phenotype"].eq("base")
        & d["ast_oxacillin_resistance_status"].eq("not_detected")
    )]
    .pipe(
        sql_merge,
        mssa_marker_cultures,
        on=reporting_culture_id,
        how="inner",
        validate="one_to_one",
    )
    .assign(
        reporting_phenotype="oxacillin_susceptible",
        organism_name=lambda d: d["organism_reporting_group"] + " — MSSA",
    )
)


### 6.3 Define the remaining published categories

Include ordinary non-*S. aureus* organisms, ESCE-R observations without positive CRE evidence, VRE in *E. faecalis* and *E. faecium*, and MRSA in *S. aureus*. CRE and MDR *P. aeruginosa* categories remain outside the main susceptibility mart.


In [ ]:
phenotype_name_suffix = {
    "esce_r": " — ESCE-R",
    "vancomycin_resistant": " — VRE",
    "oxacillin_resistant": " — MRSA",
}
ordinary_candidates = (
    stratified_sources
    .loc[lambda d: (
        (d["reporting_phenotype"].eq("base")
         & d["organism_reporting_group"].ne("Staphylococcus aureus"))
        | (d["reporting_phenotype"].eq("esce_r")
           & d["ast_cre_status"].ne("positive"))
        | (d["reporting_phenotype"].eq("vancomycin_resistant")
           & d["organism_reporting_group"].isin([
               "Enterococcus faecalis", "Enterococcus faecium"
           ]))
        | (d["reporting_phenotype"].eq("oxacillin_resistant")
           & d["organism_reporting_group"].eq("Staphylococcus aureus"))
    )]
    .assign(organism_name=lambda d: (
        d["organism_reporting_group"]
        + d["reporting_phenotype"].map(phenotype_name_suffix).fillna("")
    ))
)
stratified_candidates = pd.concat(
    [ordinary_candidates, mssa_candidates], ignore_index=True, sort=False
)


### 6.4 Select first cultures independently

After sorting by time and hash, `drop_duplicates` retains the first culture in each patient-category group. Overall *S. aureus* is selected across phenotypes in a separate pass. Keep this eligible population independent of AST availability.


In [ ]:
patient_category_id = [
    "source_version", "cohort_id", "patient_key",
    "organism_reporting_group", "reporting_phenotype",
]
first_stratified = (
    stratified_candidates
    .sort_values(["culture_time_shifted", "culture_key"], kind="stable")
    .drop_duplicates(subset=patient_category_id, keep="first")
    .reset_index(drop=True)
)
first_overall = (
    sql_merge(
        overall_cultures,
        culture_keys,
        on=culture_id,
        how="left",
        validate="many_to_one",
    )
    .assign(
        reporting_phenotype="overall",
        source_reporting_phenotype=pd.NA,
        organism_name=lambda d: d["organism_reporting_group"],
    )
    .sort_values(["culture_time_shifted", "culture_key"], kind="stable")
    .drop_duplicates(subset=patient_category_id, keep="first")
    .reset_index(drop=True)
)
selected_cultures = pd.concat(
    [first_stratified, first_overall], ignore_index=True, sort=False
)


### 6.5 Attach the matching AST scope

Join stratified selections using their source phenotype, including `base` for MSSA. Overall selections use the separately reconciled overall AST. `selected_cultures` retains cultures without AST so they can contribute to eligible denominators in the next stage.


In [ ]:
selected_columns = culture_id + [
    "cohort_id", "organism_reporting_group", "reporting_phenotype",
    "organism_name", "gram_stain_group", "culture_time_shifted", "culture_key",
]
stratified_join_id = culture_id + [
    "cohort_id", "organism_reporting_group", "source_reporting_phenotype"
]
first_stratified_ast = sql_merge(
    first_stratified.loc[:, selected_columns + ["source_reporting_phenotype"]],
    stratified_ast.rename(columns={"reporting_phenotype": "source_reporting_phenotype"}),
    on=stratified_join_id,
    how="inner",
    validate="one_to_many",
)
first_overall_ast = sql_merge(
    first_overall.loc[:, selected_columns],
    overall_ast,
    on=overall_culture_id,
    how="inner",
    validate="one_to_many",
)
selected_ast = pd.concat(
    [first_stratified_ast, first_overall_ast], ignore_index=True, sort=False
)
selected_culture_id = culture_id + [
    "cohort_id", "organism_reporting_group", "reporting_phenotype"
]


### 6.6 Check the selections

The nine expected *S. aureus* counts below come from the supplied Tableau CSV for the 2025-04-11 source release and 2023 reporting window. They are reference values, not generated notebook results. Each computed count should match when using the same inputs and window.

MRSA and MSSA need not sum to overall because each category selects its first culture independently. For example, the supplied urine populations contain 22 MRSA, 38 MSSA, and 58 overall observations.


In [ ]:
first_selection_check = pd.DataFrame({
    "population": [
        "Stratified candidate cultures",
        "First stratified cultures",
        "Overall Staphylococcus aureus candidate cultures",
        "First overall Staphylococcus aureus cultures",
    ],
    "record_count": [
        len(stratified_candidates), len(first_stratified),
        len(overall_cultures), len(first_overall),
    ],
})
staph_reference_counts = pd.DataFrame([
    ("blood", "overall", 150),
    ("blood", "oxacillin_resistant", 59),
    ("blood", "oxacillin_susceptible", 91),
    ("respiratory", "overall", 292),
    ("respiratory", "oxacillin_resistant", 34),
    ("respiratory", "oxacillin_susceptible", 258),
    ("urine", "overall", 58),
    ("urine", "oxacillin_resistant", 22),
    ("urine", "oxacillin_susceptible", 38),
], columns=["cohort_id", "reporting_phenotype", "expected_count"])
staph_selection_check = (
    selected_cultures
    .loc[lambda d: d["organism_reporting_group"].eq("Staphylococcus aureus")]
    .groupby(["cohort_id", "reporting_phenotype"], dropna=False)
    .size().rename("selected_count").reset_index()
    .merge(
        staph_reference_counts,
        on=["cohort_id", "reporting_phenotype"],
        how="outer",
        validate="one_to_one",
    )
    .assign(
        selected_count=lambda d: d["selected_count"].fillna(0).astype("int64"),
        matches_reference=lambda d: (
            d["expected_count"].notna()
            & d["selected_count"].eq(d["expected_count"])
        ),
    )
    .sort_values(["cohort_id", "reporting_phenotype"])
    .reset_index(drop=True)
)
selected_culture_counts = (
    selected_cultures
    .groupby(["cohort_id", "reporting_phenotype"], dropna=False)
    .size().rename("selected_culture_count").reset_index()
)
display(first_selection_check, staph_selection_check, selected_culture_counts)


## 7. Apply antibiotic panels and build the mart

Expand from selected cultures to approved antibiotics before attaching AST.
This keeps eligible observations with no antibiotic endpoint in the
denominator. Apply exclusions only after cohort and published phenotype
are known. A distinct excluded-pair list prevents overlapping rules from
multiplying rows.


In [ ]:
approved_pairs = (
    read_text_csv(seed_dir / "reportable_organism_antibiotic_pairs.csv")
    .assign(
        pair_order=lambda x: pd.to_numeric(x["pair_order"], errors="raise").astype("int64"),
        include_primary_dashboard=lambda x: pd.to_numeric(x["include_primary_dashboard"], errors="raise").astype("int64"),
    )
    .loc[lambda x: x["include_primary_dashboard"].eq(1) & x["review_status"].eq("reviewed")]
    [["organism_reporting_group", "cohort_id", "antibiotic", "pair_order"]]
)
exclusion_rules = (
    read_text_csv(seed_dir / "organism_antibiotic_exclusions.csv")
    .rename(columns={"cohort_id": "cohort_scope", "reporting_phenotype": "phenotype_scope"})
    [["organism_reporting_group", "antibiotic", "cohort_scope", "phenotype_scope"]]
    .drop_duplicates()
)

population_id = ["source_version", "cohort_id", "organism_reporting_group", "reporting_phenotype"]
mart_id = population_id + ["antibiotic"]

population_panels = (
    selected_cultures[population_id].drop_duplicates()
    .pipe(sql_merge, approved_pairs, on=["organism_reporting_group", "cohort_id"],
          how="inner", validate="many_to_many")
)
excluded_pairs = (
    population_panels
    .pipe(sql_merge, exclusion_rules, on=["organism_reporting_group", "antibiotic"],
          how="inner", validate="many_to_many")
    .loc[lambda x: (
        (x["cohort_scope"].eq("*") | x["cohort_id"].eq(x["cohort_scope"]))
        & (x["phenotype_scope"].eq("*") | x["reporting_phenotype"].eq(x["phenotype_scope"]))
    ).fillna(False), mart_id]
    .drop_duplicates()
)
reporting_pairs = anti_join(population_panels, excluded_pairs, on=mart_id)

eligible_grid = (
    selected_cultures[selected_culture_id + ["organism_name"]]
    .pipe(sql_merge, reporting_pairs, on=population_id, how="inner", validate="many_to_many")
)
endpoint_id = selected_culture_id + ["antibiotic"]
reporting_endpoints = sql_merge(
    eligible_grid,
    selected_ast[endpoint_id + ["resolution_status", "susceptibility", "reconciliation_reason"]],
    on=endpoint_id, how="left", validate="one_to_one",
)


### Counts, percentages, and display flags

A tested endpoint can be resolved, missing, or conflicting. No joined
endpoint means untested. Only resolved S/I/R results enter
`categorized_count`. Percent susceptible and resistant use categorized
results as their denominator; coverage uses all eligible observations.
Zero denominators yield missing percentages. Round halfway values upward
to one decimal to match SQL, then retain low-count rows with the appropriate
display flag.


In [ ]:
result_labels = {
    "susceptible": "Susceptible", "resistant": "Resistant",
    "intermediate": "Intermediate", "inconclusive": "Inconclusive",
    "synergism": "Synergism",
}
aggregation_id = [
    "source_version", "cohort_id", "organism_reporting_group", "organism_name",
    "reporting_phenotype", "antibiotic", "pair_order",
]
mart_counts = (
    reporting_endpoints
    .assign(
        is_tested=lambda x: x["resolution_status"].notna(),
        is_categorized=lambda x: (
            x["resolution_status"].eq("resolved")
            & x["susceptibility"].isin(["Susceptible", "Intermediate", "Resistant"])
        ).fillna(False),
        is_missing=lambda x: x["resolution_status"].eq("missing").fillna(False),
        is_conflict=lambda x: x["resolution_status"].eq("conflict").fillna(False),
        is_untested=lambda x: x["resolution_status"].isna(),
        **{
            f"is_{name}": (lambda x, label=label: (
                x["resolution_status"].eq("resolved") & x["susceptibility"].eq(label)
            ).fillna(False))
            for name, label in result_labels.items()
        },
    )
    .groupby(aggregation_id, as_index=False, dropna=False, observed=True, sort=False)
    .agg(
        eligible_count=("antibiotic", "size"),
        tested_endpoint_count=("is_tested", "sum"),
        categorized_count=("is_categorized", "sum"),
        susceptible_count=("is_susceptible", "sum"),
        resistant_count=("is_resistant", "sum"),
        intermediate_count=("is_intermediate", "sum"),
        inconclusive_count=("is_inconclusive", "sum"),
        synergism_count=("is_synergism", "sum"),
        missing_result_count=("is_missing", "sum"),
        conflict_result_count=("is_conflict", "sum"),
        untested_count=("is_untested", "sum"),
    )
)

def percentage_1dp(numerator, denominator):
    denominator = denominator.astype("float64").replace(0, np.nan)
    return np.floor(1000 * numerator.astype("float64") / denominator + 0.5) / 10


count_columns = [
    "eligible_count", "tested_endpoint_count", "categorized_count",
    "susceptible_count", "resistant_count", "intermediate_count",
    "inconclusive_count", "synergism_count", "missing_result_count",
    "conflict_result_count", "untested_count",
]
percentage_columns = ["categorized_result_coverage_pct", "susceptible_pct", "resistant_pct"]
date_columns = ["analysis_window_start", "analysis_window_end_exclusive"]
mart_columns = ["susceptibility_mart_key"] + aggregation_id + count_columns + percentage_columns + [
    "display_status", *date_columns, "gram_stain_group"
]

mart_susceptibility_python = (
    mart_counts
    .assign(
        susceptibility_mart_key=lambda x: make_dbt_key(x, mart_id),
        categorized_result_coverage_pct=lambda x: percentage_1dp(x["categorized_count"], x["eligible_count"]),
        susceptible_pct=lambda x: percentage_1dp(x["susceptible_count"], x["categorized_count"]),
        resistant_pct=lambda x: percentage_1dp(x["resistant_count"], x["categorized_count"]),
        display_status=lambda x: np.where(x["categorized_count"].ge(30), "display", "insufficient_data"),
        analysis_window_start=analysis_window_start.date(),
        analysis_window_end_exclusive=analysis_window_end_exclusive.date(),
    )
    .pipe(sql_merge, organism_reporting_groups, on="organism_reporting_group",
          how="left", validate="many_to_one")
    [mart_columns]
    .sort_values(population_id + ["pair_order", "antibiotic"], kind="stable")
    .reset_index(drop=True)
)

mart_build_check = pd.DataFrame([{
    "mart_rows": len(mart_susceptibility_python),
    "mart_columns": len(mart_susceptibility_python.columns),
    "reporting_populations": len(mart_susceptibility_python[population_id].drop_duplicates()),
    "duplicate_mart_keys": int(mart_susceptibility_python["susceptibility_mart_key"].duplicated().sum()),
    "eligible_balance_failures": int((
        mart_susceptibility_python["eligible_count"]
        != mart_susceptibility_python["tested_endpoint_count"] + mart_susceptibility_python["untested_count"]
    ).sum()),
    "categorized_balance_failures": int((
        mart_susceptibility_python["categorized_count"]
        != mart_susceptibility_python[["susceptible_count", "intermediate_count", "resistant_count"]].sum(axis=1)
    ).sum()),
    "endpoint_balance_failures": int((
        mart_susceptibility_python["tested_endpoint_count"]
        != mart_susceptibility_python[["categorized_count", "inconclusive_count", "synergism_count",
                                        "missing_result_count", "conflict_result_count"]].sum(axis=1)
    ).sum()),
    "missing_gram_groups": int(mart_susceptibility_python["gram_stain_group"].isna().sum()),
}])
display(mart_build_check)
display(row_counts(mart_susceptibility_python, ["display_status"], "mart_rows"))
display(mart_susceptibility_python.head(20))


## 8. Compare every field with Tableau and export

Import the supplied SQL-generated CSV as the reference. Preserve text keys,
explicitly parse numbers and dates, and recognize the lowercase `null`
missing marker. Validate columns and unique, complete natural keys before
comparing. The five identifying fields are checked through row membership;
the remaining 21 fields are compared value by value, ignoring row order.

The Python result remains independent of the reference: the reference is
first used here for validation, after all transformations are complete.


In [ ]:
tableau_reference = read_text_csv(tableau_reference_file)
numeric_columns = ["pair_order", *count_columns, *percentage_columns]

column_check = pd.DataFrame([{
    "reference_columns": len(tableau_reference.columns),
    "python_columns": len(mart_susceptibility_python.columns),
    "same_column_names": set(tableau_reference.columns) == set(mart_susceptibility_python.columns),
    "same_column_order": list(tableau_reference.columns) == list(mart_susceptibility_python.columns),
}])
display(column_check)
if not column_check["same_column_names"].iloc[0]:
    raise ValueError({
        "only_in_reference": sorted(set(tableau_reference) - set(mart_susceptibility_python)),
        "only_in_python": sorted(set(mart_susceptibility_python) - set(tableau_reference)),
    })

for column in numeric_columns:
    tableau_reference[column] = pd.to_numeric(
        tableau_reference[column].replace("null", pd.NA), errors="raise"
    ).astype("float64")
for column in date_columns:
    tableau_reference[column] = pd.to_datetime(
        tableau_reference[column].replace("null", pd.NA),
        format="%Y-%m-%d", errors="raise"
    ).dt.date

python_keys = mart_susceptibility_python[mart_id]
reference_keys = tableau_reference[mart_id]
key_check = pd.DataFrame([
    {"dataset": name, "row_count": len(keys),
     "duplicate_key_rows": int(keys.duplicated().sum()),
     "missing_key_rows": int(keys.isna().any(axis=1).sum())}
    for name, keys in [("Python mart", python_keys), ("Tableau CSV", reference_keys)]
])
display(key_check)
if key_check[["duplicate_key_rows", "missing_key_rows"]].to_numpy().any():
    raise ValueError("Resolve missing or duplicate natural keys before comparing.")

missing_from_python = anti_join(reference_keys, python_keys, on=mart_id)
extra_in_python = anti_join(python_keys, reference_keys, on=mart_id)
display(missing_from_python, extra_in_python)


### Compare values and guard the export

Normalize both sets of numeric fields to floating-point numbers before
reshaping. Compare numeric fields numerically, text exactly, and dates as
parsed dates. Paired missing values match; a missing value on only one side
is a difference. No tolerance is applied to the final one-decimal percentages.

For the supplied release and reporting window, expect 396 rows on each side,
8,316 nonkey field values, zero missing/extra rows, zero mismatches, and
`matches_reference = True`. Inspect the difference tables if a check fails.


In [ ]:
value_columns = [column for column in tableau_reference.columns if column not in mart_id]
python_comparison = mart_susceptibility_python.copy()
for column in numeric_columns:
    python_comparison[column] = python_comparison[column].astype("float64")

python_values = python_comparison.melt(
    id_vars=mart_id, value_vars=value_columns, var_name="field", value_name="python_value"
)
reference_values = tableau_reference.melt(
    id_vars=mart_id, value_vars=value_columns, var_name="field", value_name="reference_value"
)
field_comparison = (
    sql_merge(python_values, reference_values, on=mart_id + ["field"],
              how="inner", validate="one_to_one")
    .assign(values_match=lambda x: (
        x["python_value"].eq(x["reference_value"]).fillna(False)
        | (x["python_value"].isna() & x["reference_value"].isna())
    ))
)
field_differences = field_comparison.loc[
    lambda x: ~x["values_match"], mart_id + ["field", "python_value", "reference_value"]
].reset_index(drop=True)

comparison_passed = (
    len(missing_from_python) == 0 and len(extra_in_python) == 0
    and len(field_differences) == 0 and len(tableau_reference) > 0
    and len(field_comparison) == len(tableau_reference) * len(value_columns)
)
comparison_check = pd.DataFrame([{
    "reference_rows": len(tableau_reference),
    "python_rows": len(mart_susceptibility_python),
    "missing_from_python": len(missing_from_python),
    "extra_in_python": len(extra_in_python),
    "nonkey_fields_compared": len(value_columns),
    "field_values_compared": len(field_comparison),
    "field_mismatches": len(field_differences),
    "matches_reference": comparison_passed,
}])
display(comparison_check, field_differences.head(30))


In [ ]:
if not comparison_passed:
    raise ValueError("The marts differ. Review the comparison tables before exporting.")
if output_file.resolve() == tableau_reference_file.resolve():
    raise ValueError("Choose a separate output path for the Python mart.")
failure_columns = [
    "duplicate_mart_keys", "eligible_balance_failures", "categorized_balance_failures",
    "endpoint_balance_failures", "missing_gram_groups"
]
if mart_build_check[failure_columns].to_numpy().any():
    raise ValueError("Resolve the mart construction failures before exporting.")

output_file.parent.mkdir(parents=True, exist_ok=True)
mart_susceptibility_python[list(tableau_reference.columns)].to_csv(
    output_file, index=False, na_rep="null", encoding="utf-8"
)
export_check = pd.DataFrame([{
    "output_file": str(output_file),
    "exported_rows": len(mart_susceptibility_python),
    "exported_columns": len(mart_susceptibility_python.columns),
    "matches_tableau_reference": comparison_passed,
}])
display(export_check)


When the comparison passes, this notebook demonstrates the same end-to-end
transformation as the validated R pipeline. The generated CSV is
`data/processed/mart_susceptibility_python.csv`. CSV numeric formatting and
row ordering can differ while the typed data values remain equal.

Save the notebook after running all cells so your portfolio includes the
actual Python outputs and validation results. Expected checkpoint numbers
in the narrative should not substitute for the executed comparison.
